<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/4%E1%84%8C%E1%85%AE%E1%84%8E%E1%85%A1_%E1%84%90%E1%85%A6%E1%86%A8%E1%84%89%E1%85%B3%E1%84%90%E1%85%B3_%E1%84%87%E1%85%AE%E1%86%AB%E1%84%85%E1%85%B2_%E1%84%8F%E1%85%A9%E1%84%85%E1%85%A2%E1%86%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 4주차 | 텍스트 분류 — 긍정/부정 자동 판단

> **오늘의 방식**
> 1. 핵심 코드를 **먼저 실행**해서 결과를 확인합니다
> 2. 결과 숫자가 무엇을 의미하는지 **해석**합니다
> 3. 각 단계가 **왜 이렇게 작동하는지** 배웁니다

```
리뷰 텍스트 → 전처리 → TF-IDF → 머신러닝 모델 → 긍정 😊 / 부정 😞
```
---

## 🚀 STEP 0. 먼저 전체 코드를 실행해봅시다

이론 설명 전에 전체 파이프라인을 한 번 돌려봅니다.
지금은 이해하지 못해도 괜찮습니다. **결과가 나오는 것**을 먼저 경험합니다.

> ⬇️ 아래 셀을 **순서대로** 실행하세요 (Shift+Enter)

In [1]:
# 필요한 도구 설치 및 불러오기
!pip install scikit-learn --quiet

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split # 데이터 분할
from sklearn.naive_bayes import MultinomialNB # 나이브 베이즈
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀분석
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# 기타 결과 해석
print("✅ 준비 완료!")

✅ 준비 완료!


In [2]:
# 학습 데이터: 긍정 25개 + 부정 25개 = 총 50개
# 주의 할점 -> 1번째 긍정리뷰가 진짜 긍정? 부정?
# 정말로 이 문장이 한가지 의미를 내포하고 있는지?
긍정_리뷰 = [
    "치킨이 정말 바삭하고 맛있어요 소스도 최고예요",
    "배달이 엄청 빠르고 포장도 꼼꼼했어요 재주문 할게요",
    "피자 도우가 쫄깃하고 치즈가 넘쳐요 완전 맛집",
    "서비스가 친절하고 음식도 맛있어요 강력 추천",
    "가성비 최고예요 양도 많고 맛도 좋아요",
    "국물이 진하고 깊은 맛이에요 또 시킬게요",
    "고기가 부드럽고 신선해요 냄새도 안 나요",
    "빠른 배달에 음식 맛도 훌륭해요 감사합니다",
    "포장이 깔끔하고 음식이 따뜻하게 왔어요",
    "사장님이 너무 친절하세요 음식도 맛있고요",
    "면이 쫄깃하고 국물이 시원해요 자주 시킬게요",
    "닭고기가 촉촉하고 양념이 잘 배어있어요",
    "해산물이 신선하고 볶음밥도 맛있어요",
    "두부가 부드럽고 찌개 간이 딱 맞아요",
    "냉면 육수가 시원하고 고명이 푸짐해요",
    "된장찌개 맛이 구수하고 밥이 윤기나요",
    "떡볶이가 쫄깃하고 소스가 매콤달콤해요",
    "샐러드가 신선하고 드레싱이 맛있어요",
    "갈비찜이 부드럽고 양이 많아서 만족해요",
    "재료가 신선하고 맛이 깔끔해서 자주 시켜요",
    "치즈가 풍부하고 빵이 바삭해서 맛있었어요",
    "배달 포장이 완벽하고 음식도 따뜻하게 왔어요",
    "양념이 골고루 배어있고 육즙이 살아있어요",
    "순두부찌개 국물이 얼큰하고 두부가 신선해요",
    "파스타 면이 탱탱하고 소스가 진해요",
]

부정_리뷰 = [
    "배달이 너무 늦어서 음식이 다 식었어요",
    "치킨이 눅눅하고 기름기가 너무 많아요",
    "포장이 엉망이라 국물이 다 쏟아졌어요",
    "서비스가 불친절하고 음식도 별로였어요",
    "양이 너무 적고 가격은 비싸요 실망이에요",
    "음식에서 이상한 냄새가 났어요 위생이 걱정돼요",
    "주문한 거랑 다른 음식이 왔어요 환불하고 싶어요",
    "맛이 너무 짜고 자극적이에요 먹기 힘들었어요",
    "배달 시간이 두 시간이나 걸렸어요 너무해요",
    "재료가 신선하지 않은 것 같아요 다시는 안 시켜요",
    "면이 퉁퉁 불어있고 국물이 싱거워요",
    "고기가 질기고 냄새가 심하게 나요",
    "두부가 오래된 것 같고 찌개가 너무 짜요",
    "냉면이 식어있고 육수 맛이 이상해요",
    "된장찌개에서 쉰 냄새가 나요 못 먹겠어요",
    "떡이 딱딱하고 소스가 너무 매워서 못 먹겠어요",
    "샐러드 재료가 시들어있고 드레싱이 상한 것 같아요",
    "파스타가 너무 짜고 면이 퍼져있어요",
    "치즈가 굳어있고 빵이 눅눅해서 실망했어요",
    "음식이 차갑게 왔고 포장도 터져있었어요",
    "양념이 안 배어있고 육즙이 없어서 퍽퍽해요",
    "재료가 변질된 것 같아서 먹다가 버렸어요",
    "배달원이 불친절하고 음식도 엉망이었어요",
    "가격 대비 질이 너무 낮아요 다시는 안 시킬게요",
    "조리가 덜 됐는지 속이 안 익었어요",
]

X = 긍정_리뷰 + 부정_리뷰   # 전체 리뷰 - X 데이터
y = [1]*25 + [0]*25          # 정답 (1=긍정, 0=부정)

print(f"전체 데이터: {len(X)}개  (긍정 {sum(y)}개 / 부정 {len(y)-sum(y)}개)")

전체 데이터: 50개  (긍정 25개 / 부정 25개)


In [3]:
# 전처리 함수
def 전처리(텍스트):
    # 한글, 영어, 숫자, 공백이 아닌것은 모두 제거해라 ' '으로
    # 깔끔한 문자를 가진 문장으로 만드는 것
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', ' ', 텍스트)
    return re.sub(r'\s+', ' ', 텍스트).strip() # \s -> 공백문자 전체 + 1개 이상 .strip() 은 양쪽 공백 제거

In [4]:
# 리스트 컴프리핸션으로 문장에 대해 반복 처리
X = [전처리(r) for r in X]

In [5]:
# 데이터 분리
# 1. 새로운 리뷰에 대해서 잘 맞추는지 못맞추는지 검증
# 2. 학습데이터와 테스트데이터를 나눠서 골고루 잘 구성되는지 확인
# X데이터, y데이터, y데이터 비율, 데이터 고정
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [6]:
# TF-IDF 변환
# 테스트 데이터를 미리 학습해서 train이 미리 테스트할 내용을 알지 못하게하는 것이 중요
# 데이터 누수 -> 미래의 일 또는 알지못하는 것에 대해서 미리 학습을 해서 편향이 생기지 않게 하기 위함

변환기 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4))
X_train_vec = 변환기.fit_transform(X_train) # 훈련을 할때에는 fit_transform 학습
X_test_vec = 변환기.transform(X_test) #테스트데이터는 transform 변환

In [7]:
print(X_train_vec) # 희소행렬 37 문서 / 1157개의 단어

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1915 stored elements and shape (37, 1157)>
  Coords	Values
  (0, 367)	0.12583386020479412
  (0, 1115)	0.12583386020479412
  (0, 980)	0.14376230254527628
  (0, 942)	0.1123368027334113
  (0, 368)	0.12583386020479412
  (0, 1118)	0.14376230254527628
  (0, 981)	0.14376230254527628
  (0, 370)	0.14376230254527628
  (0, 1119)	0.14376230254527628
  (0, 44)	0.174411177162674
  (0, 468)	0.174411177162674
  (0, 477)	0.174411177162674
  (0, 1130)	0.07486585403674512
  (0, 430)	0.05321951023304988
  (0, 45)	0.174411177162674
  (0, 469)	0.174411177162674
  (0, 478)	0.174411177162674
  (0, 1131)	0.07486585403674512
  (0, 46)	0.174411177162674
  (0, 470)	0.174411177162674
  (0, 479)	0.174411177162674
  (0, 285)	0.10324669411185801
  (0, 936)	0.10324669411185801
  (0, 827)	0.14376230254527628
  (0, 286)	0.10324669411185801
  :	:
  (36, 488)	0.1543821343155552
  (36, 58)	0.1543821343155552
  (36, 925)	0.1543821343155552
  (36, 277)	0.154382134

In [8]:
# 전처리 완료 후
# 깔끔한 문장 -> 깔끔한 단어 / 형태소 -> 특정 단어가 나왔을때 긍정/ 특정 단어가 나왔을때 부정 예측

모델 = MultinomialNB() # 모델을 선정해서 fit을 통해 학습을 한다.
모델.fit(X_train_vec, y_train) # 최종 변환된 데이터, 정답
예측 = 모델.predict(X_test_vec) # 현재 미래의 데이터라고 가정하고 test데이터를 통해 검증을 한다.



In [9]:
print(y_test) # 실제값
print(예측) # 예측값

[1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0]
[1 0 0 1 1 0 1 0 0 0 1 1 0]


In [10]:
# 평가
print(accuracy_score(y_test, 예측))

0.7692307692307693


In [11]:
print(classification_report(y_test, 예측))

              precision    recall  f1-score   support

           0       0.86      0.75      0.80         8
           1       0.67      0.80      0.73         5

    accuracy                           0.77        13
   macro avg       0.76      0.78      0.76        13
weighted avg       0.78      0.77      0.77        13



In [12]:
# ══════════════════════════════════════════
# 핵심 파이프라인 5단계
# ══════════════════════════════════════════

# ① 전처리
def 전처리(텍스트):
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', ' ', 텍스트)
    return re.sub(r'\s+', ' ', 텍스트).strip()

X = [전처리(r) for r in X]

# ② 데이터 분리 (훈련 75% / 테스트 25%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

# ③ TF-IDF 변환
변환기 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_train_vec = 변환기.fit_transform(X_train)   # 훈련만 fit!
X_test_vec  = 변환기.transform(X_test)         # 테스트는 transform만!

# ④ 학습 + 예측 (나이브 베이즈)
모델 = MultinomialNB()
모델.fit(X_train_vec, y_train)
예측 = 모델.predict(X_test_vec)

# ⑤ 평가
print(f"정확도: {accuracy_score(y_test, 예측):.2%}")
print()
print(classification_report(y_test, 예측, target_names=["부정", "긍정"]))

정확도: 76.92%

              precision    recall  f1-score   support

          부정       0.86      0.75      0.80         8
          긍정       0.67      0.80      0.73         5

    accuracy                           0.77        13
   macro avg       0.76      0.78      0.76        13
weighted avg       0.78      0.77      0.77        13



---
## 📊 STEP 1. 결과 해석

코드를 실행했으니, 결과 숫자들이 무엇을 의미하는지 읽어봅니다.

### 1-1. 정확도 (Accuracy)

```
정확도: 84.62%
→ 테스트 리뷰 13개 중 11개를 올바르게 분류
```

> 💡 **현실 팁**  
> 정확도 70%도 실전에서 충분히 유용합니다.  
> 1만 개 리뷰 중 7,000개를 자동 처리하고, 나머지 3,000개만 사람이 검토하면 됩니다.  
> 정확도 100%를 목표로 하기보다 "어느 정도면 실용적인가"를 먼저 생각합니다.

### 1-2. classification_report 읽는 법

```
              precision    recall  f1-score   support
        부정       0.86      0.86      0.86         7  ← 실제 부정 7개
        긍정       0.83      0.83      0.83         6  ← 실제 긍정 6개
    accuracy                           0.85        13  ← 전체 정확도
   macro avg       0.85      0.85      0.85        13  ← 단순 평균
weighted avg       0.85      0.85      0.85        13  ← 개수 비례 평균
```

| 열 | 의미 |
|---|---|
| `support` | 테스트에 실제로 있는 개수 |
| `precision` | 긍정이라 예측한 것 중 실제 긍정 비율 |
| `recall` | 실제 긍정 중 긍정으로 찾아낸 비율 |
| `f1-score` | precision과 recall의 균형 점수 |

In [13]:
# 어떤 리뷰를 맞추고 틀렸는지 직접 확인
print("번호  실제   예측   정오")
print("-" * 26)

for i in range(len(y_test)):
    실제  = "긍정" if y_test[i] == 1 else "부정"
    예측값 = "긍정" if 예측[i]   == 1 else "부정"
    정오  = "✅" if y_test[i] == 예측[i] else "❌"
    print(f"  {i+1:2}번   {실제}   {예측값}   {정오}")

번호  실제   예측   정오
--------------------------
   1번   긍정   긍정   ✅
   2번   부정   부정   ✅
   3번   부정   부정   ✅
   4번   부정   긍정   ❌
   5번   긍정   긍정   ✅
   6번   부정   부정   ✅
   7번   부정   긍정   ❌
   8번   부정   부정   ✅
   9번   부정   부정   ✅
  10번   긍정   부정   ❌
  11번   긍정   긍정   ✅
  12번   긍정   긍정   ✅
  13번   부정   부정   ✅


> **확인 포인트**  
> ❌ 가 나온 리뷰를 X_test에서 직접 찾아 읽어보세요.  
> 모델이 왜 틀렸는지 감이 잡힐 것입니다.

In [14]:
# 각 리뷰의 긍정/부정 확률도 확인
확률 = 모델.predict_proba(X_test_vec)

print("번호  부정확률  긍정확률  예측    실제   정오")
print("-" * 48)
for i, (부정_p, 긍정_p) in enumerate(확률):
    예측값 = "긍정 😊" if 예측[i] == 1 else "부정 😞"
    실제   = "긍정" if y_test[i] == 1 else "부정"
    정오   = "✅" if 예측[i] == y_test[i] else "❌"
    print(f"  {i+1:2}번   {부정_p:.4f}   {긍정_p:.4f}   {예측값}  {실제}  {정오}")

번호  부정확률  긍정확률  예측    실제   정오
------------------------------------------------
   1번   0.3190   0.6810   긍정 😊  긍정  ✅
   2번   0.5396   0.4604   부정 😞  부정  ✅
   3번   0.5906   0.4094   부정 😞  부정  ✅
   4번   0.3221   0.6779   긍정 😊  부정  ❌
   5번   0.4841   0.5159   긍정 😊  긍정  ✅
   6번   0.5403   0.4597   부정 😞  부정  ✅
   7번   0.4672   0.5328   긍정 😊  부정  ❌
   8번   0.5725   0.4275   부정 😞  부정  ✅
   9번   0.5921   0.4079   부정 😞  부정  ✅
  10번   0.5217   0.4783   부정 😞  긍정  ❌
  11번   0.3132   0.6868   긍정 😊  긍정  ✅
  12번   0.3549   0.6451   긍정 😊  긍정  ✅
  13번   0.5399   0.4601   부정 😞  부정  ✅


> **확률 해석**  
> - 긍정 확률 **0.9 이상** → 모델이 강하게 확신  
> - 긍정 확률 **0.5 근처** → 모델이 헷갈리는 경우  
> - 긍정 확률 **0.1 이하** → 부정이라고 강하게 확신

---
## 🔍 STEP 2. 이론 ① — 전처리

핵심 코드에서 사용한 전처리 함수를 자세히 살펴봅니다.

### 코드 다시 보기

```python
def 전처리(텍스트):
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', ' ', 텍스트)
    return re.sub(r'\s+', ' ', 텍스트).strip()
```

**패턴 `[^가-힣a-zA-Z0-9 ]` 읽기:**

| 부분 | 의미 |
|------|------|
| `[^...]` | 이것이 **아닌** 것 |
| `가-힣` | 한글 전체 |
| `a-zA-Z` | 영어 대소문자 |
| `0-9` | 숫자 |
| ` ` | 공백 |

→ "한글, 영어, 숫자, 공백이 아닌 모든 것"을 공백으로 바꿉니다.

> 💡 **현실 팁**  
> KoNLPy와 char n-gram 중 어느 게 더 나을지 모르면 둘 다 해보고 성능이 높은 것을 선택합니다.  
> 이것을 **Ablation Study**라고 합니다. 정답은 데이터마다 다릅니다.

In [15]:
# 전처리 효과 직접 확인
테스트_문장들 = [
    "치킨이 진짜 바삭하고 맛있어요!!! ㅋㅋ",
    "배달이 너무 느려요ㅠㅠ http://review.com",
    "음식이   너무   짜요...",
]

print("전처리 효과:")
print("-" * 55)
for 문장 in 테스트_문장들:
    print(f"  입력: '{문장}'")
    print(f"  출력: '{전처리(문장)}'")
    print()

전처리 효과:
-------------------------------------------------------
  입력: '치킨이 진짜 바삭하고 맛있어요!!! ㅋㅋ'
  출력: '치킨이 진짜 바삭하고 맛있어요'

  입력: '배달이 너무 느려요ㅠㅠ http://review.com'
  출력: '배달이 너무 느려요 http review com'

  입력: '음식이   너무   짜요...'
  출력: '음식이 너무 짜요'



---
## 🔍 STEP 3. 이론 ② — 데이터 분리

### 왜 나눠야 할까?

**시험 비유:**
```
[나쁜 방법]
  시험 문제를 미리 알고 공부 → 시험 100점
  → 진짜 실력이 아님

[올바른 방법]
  시험 문제 모르고 공부 → 진짜 실력 측정

머신러닝도 동일:
  훈련 데이터: 모델이 공부하는 데이터  (37개)
  테스트 데이터: 시험 문제, 학습에 절대 사용 금지! (13개)
```

### random_state=42 의미

```python
# random_state 없이: 실행할 때마다 다른 결과
# random_state=42: 항상 같은 방식으로 분리 → 재현 가능
# 42는 IT 커뮤니티 농담("인생의 답=42")에서 비롯된 관례
```

### ⚠️ 데이터 누수 (Data Leakage) — 가장 흔한 실수

> 💡 **현실 팁**  
> 성능이 갑자기 너무 좋게 나오면 데이터 누수를 의심하세요.  
> 실무에서도 매우 자주 발생하는 실수입니다.

In [16]:
# 올바른 순서 vs 잘못된 순서 비교

print("✅ 올바른 순서:")
print("   ① 훈련/테스트 분리 먼저")
print("   ② 훈련 데이터로만 vectorizer.fit()")
print("   ③ 테스트는 transform()만")
print()
print("❌ 잘못된 순서 (데이터 누수):")
print("   ① 전체 50개로 vectorizer.fit()  ← 테스트 정보가 IDF에 섞임!")
print("   ② 그 다음 훈련/테스트 분리")
print("   → 성능이 실제보다 좋게 측정됨 (속는 것!)")
print()

# 분리 결과 확인
print(f"훈련 데이터: {len(X_train)}개  (전체의 75%)")
print(f"테스트 데이터: {len(X_test)}개   (전체의 25%)")
print()
print(f"훈련 — 긍정: {sum(y_train)}개, 부정: {len(y_train)-sum(y_train)}개")
print(f"테스트 — 긍정: {sum(y_test)}개, 부정: {len(y_test)-sum(y_test)}개")

✅ 올바른 순서:
   ① 훈련/테스트 분리 먼저
   ② 훈련 데이터로만 vectorizer.fit()
   ③ 테스트는 transform()만

❌ 잘못된 순서 (데이터 누수):
   ① 전체 50개로 vectorizer.fit()  ← 테스트 정보가 IDF에 섞임!
   ② 그 다음 훈련/테스트 분리
   → 성능이 실제보다 좋게 측정됨 (속는 것!)

훈련 데이터: 37개  (전체의 75%)
테스트 데이터: 13개   (전체의 25%)

훈련 — 긍정: 20개, 부정: 17개
테스트 — 긍정: 5개, 부정: 8개


---
## 🔍 STEP 4. 이론 ③ — TF-IDF 변환

### char n-gram 방식

```python
변환기 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
```

**n-gram이란?** 연속된 n개의 문자 조합입니다.

```
"바삭하고" → char n-gram 분해:
  2글자: "바삭", "삭하", "하고"
  3글자: "바삭하", "삭하고"
  4글자: "바삭하고"
```

**한국어에서 효과적인 이유:**

```
한국어는 같은 의미도 형태가 다양합니다:
  맛있다 / 맛있어요 / 맛있었어요 / 맛있고
  
공통 패턴 "맛있"이 항상 등장합니다!
→ KoNLPy 없이도 형태 변화를 자연스럽게 처리
```

> 💡 **현실 팁**  
> `fit_transform`과 `transform`의 구분은 sklearn의 모든 전처리 도구에 동일하게 적용됩니다.  
> `StandardScaler`, `LabelEncoder` 등도 훈련 데이터에만 `fit()`을 씁니다.  
> 이 패턴을 한 번 익히면 어떤 도구에서도 같은 방식으로 씁니다.

In [17]:
# TF-IDF 행렬 확인
print(f"훈련 TF-IDF 행렬: {X_train_vec.shape}")
print(f"  → {X_train_vec.shape[0]}개 리뷰 × {X_train_vec.shape[1]}개 특성(n-gram 패턴)")
print()
print(f"테스트 TF-IDF 행렬: {X_test_vec.shape}")
print()

# 추출된 특성(n-gram 패턴) 일부 확인
특성들 = 변환기.get_feature_names_out()
print(f"전체 특성 수: {len(특성들)}개")
print(f"특성 예시 (앞 20개): {list(특성들[:20])}")

훈련 TF-IDF 행렬: (37, 1157)
  → 37개 리뷰 × 1157개 특성(n-gram 패턴)

테스트 TF-IDF 행렬: (13, 1157)

전체 특성 수: 1157개
특성 예시 (앞 20개): [' 가', ' 가격', ' 가격은', ' 갈', ' 갈비', ' 갈비찜', ' 감', ' 감사', ' 감사합', ' 강', ' 강력', ' 강력 ', ' 같', ' 같아', ' 같아서', ' 같아요', ' 거', ' 거랑', ' 거랑 ', ' 걸']


---
## 🔍 STEP 5. 이론 ④ — 두 가지 분류기

### 나이브 베이즈 (MultinomialNB)

**핵심 아이디어:** 각 패턴이 긍정/부정에 등장하는 확률을 학습합니다.

```
학습 결과 예시:
  패턴       긍정 확률   부정 확률
  "맛있"      80%         10%
  "바삭"      75%          8%
  "식었"       5%         70%
  "별로"       3%         65%

새 리뷰 "바삭 맛있다":
  P(긍정) ∝ 0.75 × 0.80 = 0.60  ← 높음!
  P(부정) ∝ 0.08 × 0.10 = 0.008 ← 낮음
  → 긍정 분류
```

"나이브(순진한)" 가정: 각 단어가 서로 독립적이라고 가정합니다.  
현실과 맞지 않지만, 계산이 단순하고 놀랍게도 잘 작동합니다.

### 로지스틱 회귀 (LogisticRegression)

**핵심 아이디어:** 각 패턴에 가중치를 부여해서 점수를 계산합니다.

```
점수 = w("맛있")×2.1 + w("바삭")×1.8 + w("식었")×(-2.3) + ...
점수 → 시그모이드 함수 → 0~1 확률 → 분류
```

**강점:** 어떤 패턴이 판단에 영향을 줬는지 가중치로 확인 가능합니다.

> 💡 **현실 팁**  
> 실전에서는 두 모델 모두 돌려보고 성능이 좋은 것을 선택합니다.  
> "단순한 모델이 항상 나쁜 것은 아니다"는 머신러닝의 중요한 교훈입니다.

In [18]:
# ── 나이브 베이즈 결과 (이미 실행됨) ──
nb_acc = accuracy_score(y_test, 예측)
print(f"나이브 베이즈 정확도: {nb_acc:.2%}  {'█'*int(nb_acc*20)}")

나이브 베이즈 정확도: 76.92%  ███████████████


In [19]:
# ── 로지스틱 회귀도 비교해봅시다 ──
모델_LR = LogisticRegression(max_iter=1000, random_state=42)
모델_LR.fit(X_train_vec, y_train)
예측_LR = 모델_LR.predict(X_test_vec)

lr_acc = accuracy_score(y_test, 예측_LR)
print(f"로지스틱 회귀 정확도: {lr_acc:.2%}  {'█'*int(lr_acc*20)}")
print()

# 둘 중 어느 쪽이 더 좋은지 비교
if nb_acc >= lr_acc:
    print("→ 이 데이터에서는 나이브 베이즈가 더 좋거나 동일합니다")
else:
    print("→ 이 데이터에서는 로지스틱 회귀가 더 좋습니다")

로지스틱 회귀 정확도: 84.62%  ████████████████

→ 이 데이터에서는 로지스틱 회귀가 더 좋습니다


---
## 🔍 STEP 6. 이론 ⑤ — 성능 평가

### 정확도만으로는 부족한 경우

```
예시: 리뷰 100개 중 긍정 95개, 부정 5개

모델이 모든 것을 "긍정"이라고 예측하면?
  → 정확도 = 95/100 = 95%  ← 높아 보이지만
  → 부정 리뷰를 하나도 못 잡아냄!
```

### 혼동 행렬 (Confusion Matrix)

```
                  예측: 부정(0)   예측: 긍정(1)
실제: 부정(0)  [   TN (맞음)  |   FP (오탐)  ]
실제: 긍정(1)  [   FN (미탐)  |   TP (맞음)  ]
```

### Precision / Recall / F1

```
           TP
Precision = ─────────    "긍정 예측 중 진짜 긍정 비율"
           TP + FP        → 스팸 필터에서 중요 (정상 메일 스팸함 금지)

          TP
Recall = ─────────        "실제 긍정 중 놓치지 않은 비율"
         TP + FN           → 의료 진단에서 중요 (암 환자 놓치면 안 됨)

         2 × P × R
F1 = ──────────────       "두 지표의 균형"
         P + R              → 일반적인 상황에서 기본 지표
```

> 💡 **현실 팁**  
> `weighted avg`는 각 클래스의 개수(support)를 고려한 평균입니다.  
> 클래스 불균형(긍정 90개 + 부정 10개)이 있을 때 `macro avg`보다 신뢰할 수 있습니다.

In [20]:
# 혼동 행렬 확인
cm = confusion_matrix(y_test, 예측)

print("혼동 행렬 (나이브 베이즈):")
print()
print("              예측:부정   예측:긍정")
print(f"  실제:부정      {cm[0][0]:3}        {cm[0][1]:3}")
print(f"  실제:긍정      {cm[1][0]:3}        {cm[1][1]:3}")
print()
print(f"  TN(✅ 부정→부정): {cm[0][0]}개")
print(f"  FP(❌ 부정→긍정): {cm[0][1]}개  ← 불만 고객을 만족으로 착각")
print(f"  FN(❌ 긍정→부정): {cm[1][0]}개  ← 만족 고객을 불만으로 착각")
print(f"  TP(✅ 긍정→긍정): {cm[1][1]}개")

혼동 행렬 (나이브 베이즈):

              예측:부정   예측:긍정
  실제:부정        6          2
  실제:긍정        1          4

  TN(✅ 부정→부정): 6개
  FP(❌ 부정→긍정): 2개  ← 불만 고객을 만족으로 착각
  FN(❌ 긍정→부정): 1개  ← 만족 고객을 불만으로 착각
  TP(✅ 긍정→긍정): 4개


In [21]:
# 두 모델 상세 성능 비교
print("=" * 42)
print("나이브 베이즈 상세 성능")
print("=" * 42)
print(classification_report(y_test, 예측,
                              target_names=["부정", "긍정"],
                              zero_division=0))

print("=" * 42)
print("로지스틱 회귀 상세 성능")
print("=" * 42)
print(classification_report(y_test, 예측_LR,
                              target_names=["부정", "긍정"],
                              zero_division=0))

나이브 베이즈 상세 성능
              precision    recall  f1-score   support

          부정       0.86      0.75      0.80         8
          긍정       0.67      0.80      0.73         5

    accuracy                           0.77        13
   macro avg       0.76      0.78      0.76        13
weighted avg       0.78      0.77      0.77        13

로지스틱 회귀 상세 성능
              precision    recall  f1-score   support

          부정       1.00      0.75      0.86         8
          긍정       0.71      1.00      0.83         5

    accuracy                           0.85        13
   macro avg       0.86      0.88      0.85        13
weighted avg       0.89      0.85      0.85        13



---
## 🔍 STEP 7. 가중치 분석 — 모델이 본 근거

로지스틱 회귀의 강점: 어떤 패턴이 판단에 영향을 줬는지 확인할 수 있습니다.

```python
가중치[i] > 0  → 이 패턴이 있으면 긍정 쪽으로 판단
가중치[i] < 0  → 이 패턴이 있으면 부정 쪽으로 판단
가중치[i] ≈ 0  → 판단에 거의 영향 없음
```

> 💡 **현실 팁**  
> 가중치가 이상하게 나왔다면 (예: "맛없다"가 긍정 패턴에 포함) 데이터 레이블에 문제가 있을 가능성이 높습니다.  
> 가중치 확인은 모델 디버깅의 첫 번째 단계입니다.

In [22]:
# 의미 없는 n-gram 패턴 필터
# char_wb n-gram 특성상 조사·어미·공백만 있는 패턴이 섞입니다
# → 실제 의미 있는 단어 패턴만 골라냅니다

어미_조사 = {
    '어요','해요','하고','게요','이고','이다','이에','있고','어있','고요',
    '이요','네요','에요','라고','으로','에서','지만','는데','으면','아요',
    '겠어','하여','하며','이며','이라','이나','라서','아서','어서','해서',
    '하면','이면','라면','에게','으며','으나','이야','이죠','했어','됐어',
    '갔어','왔어','있어','없어','같아','싶어','봐요','줘요','하죠','이죠',
    '라죠','하네','이네','라네','다네','했네','됐네',
}

def 의미있는_패턴(p):
    """
    의미 있는 n-gram 패턴인지 판단합니다.
    걸러내는 경우:
      - 공백 제거 후 1글자 이하 (예: ' 불', ' 다')
      - 순수 조사·어미만으로 구성 (예: '어요', '하고', '해요')
      - 끝 공백 포함 중복 패턴 (예: '어요 ' → '어요'와 동일)
    """
    p_clean = p.strip()
    if len(p_clean) <= 1:                  # 1글자 이하
        return False
    if p_clean in 어미_조사:               # 순수 어미·조사
        return False
    if p.endswith(' ') and p.strip() in 어미_조사:   # 공백 포함 어미
        return False
    return True

print("의미 없는 패턴 필터 준비 완료!")
print()
print("예시:")
테스트 = ['어요', '어요 ', ' 맛있', '바삭', '하고', '치킨', '식었', ' 불', '불친절']
for p in 테스트:
    결과 = "✅ 유지" if 의미있는_패턴(p) else "❌ 제거"
    print(f"  {결과}: '{p}'")

의미 없는 패턴 필터 준비 완료!

예시:
  ❌ 제거: '어요'
  ❌ 제거: '어요 '
  ✅ 유지: ' 맛있'
  ✅ 유지: '바삭'
  ❌ 제거: '하고'
  ✅ 유지: '치킨'
  ✅ 유지: '식었'
  ❌ 제거: ' 불'
  ✅ 유지: '불친절'


In [23]:
feature_names = 변환기.get_feature_names_out()
가중치 = 모델_LR.coef_[0]

# 의미 있는 패턴만 필터링
의미있는_인덱스 = [i for i, p in enumerate(feature_names) if 의미있는_패턴(p)]
필터된_feature  = feature_names[의미있는_인덱스]
필터된_가중치   = 가중치[의미있는_인덱스]

# 긍정에 강한 패턴 TOP 10
긍정_순서 = 필터된_가중치.argsort()[-10:][::-1]

print("긍정에 강한 패턴 TOP 10  (+ 클수록 긍정에 영향):")
print("-" * 42)
for i in 긍정_순서:
    막대 = "█" * min(int(필터된_가중치[i] * 3), 20)
    print(f"  '{필터된_feature[i]}':  {필터된_가중치[i]:+.3f}  {막대}")

긍정에 강한 패턴 TOP 10  (+ 클수록 긍정에 영향):
------------------------------------------
  ' 맛있':  +0.184  
  '맛있':  +0.184  
  '쫄깃하고':  +0.134  
  '쫄깃하':  +0.134  
  '깃하고 ':  +0.134  
  '깃하고':  +0.134  
  '깃하':  +0.134  
  '쫄깃':  +0.134  
  ' 쫄깃하':  +0.134  
  ' 쫄깃':  +0.134  


In [24]:
# 부정에 강한 패턴 TOP 10
부정_순서 = 필터된_가중치.argsort()[:10]

print("부정에 강한 패턴 TOP 10  (- 클수록 부정에 영향):")
print("-" * 42)
for i in 부정_순서:
    막대 = "█" * min(int(abs(필터된_가중치[i]) * 3), 20)
    print(f"  '{필터된_feature[i]}':  {필터된_가중치[i]:+.3f}  {막대}")

부정에 강한 패턴 TOP 10  (- 클수록 부정에 영향):
------------------------------------------
  '망이':  -0.176  
  '어있고':  -0.158  
  '어있고 ':  -0.158  
  ' 너무':  -0.151  
  '너무':  -0.151  
  ' 재료':  -0.142  
  ' 재료가':  -0.142  
  '재료':  -0.142  
  '재료가':  -0.142  
  '료가':  -0.142  


> **확인 포인트**  
> 긍정 패턴에 "맛있", "바삭", "친절", "최고" 같은 표현이 나왔나요?  
> 부정 패턴에 "식었", "별로", "불친", "늦어" 같은 표현이 나왔나요?  
> 사람이 읽어도 납득이 되면 모델이 잘 학습한 것입니다.

---
## 🔍 STEP 8. 새 리뷰 직접 예측하기

### transform만 사용하는 이유

```
새 리뷰가 들어올 때마다 fit()을 다시 하면?
  → 어휘와 IDF가 바뀜
  → 훈련 때와 다른 기준으로 변환
  → 모델이 엉뚱한 결과를 냄

transform만 사용하면?
  → 훈련 때 학습한 기준 유지
  → 일관된 결과 보장
```

In [25]:
# 예측 함수 만들기
def 예측하기(새_리뷰, 모델):
    처리   = 전처리(새_리뷰)                  # ① 전처리
    벡터   = 변환기.transform([처리])          # ② transform만! (fit 하면 안 됨)
    예측값 = 모델.predict(벡터)[0]             # ③ 예측
    확률값 = 모델.predict_proba(벡터)[0]       # ④ 확률
    결과   = "긍정 😊" if 예측값 == 1 else "부정 😞"
    return 결과, 확률값[1]                     # 긍정 확률 반환

print("예측 함수 준비 완료!")

예측 함수 준비 완료!


In [26]:
# 새 리뷰 5개 예측
새_리뷰들 = [
    "치킨이 바삭하고 소스가 완벽해요 최고예요",        # 명확한 긍정
    "배달이 세 시간이나 걸렸고 음식도 식어있었어요",    # 명확한 부정
    "맛은 보통이에요 특별하지는 않아요",               # 중립 (애매)
    "포장은 좋았는데 음식 맛이 별로였어요",            # 혼합 감성
    "가격 대비 양도 많고 배달도 빠르고 만족해요",      # 긍정
]

print("새 리뷰 예측 결과:")
print("=" * 60)
for 리뷰 in 새_리뷰들:
    결과_nb, 확률_nb = 예측하기(리뷰, 모델)
    결과_lr, 확률_lr = 예측하기(리뷰, 모델_LR)
    print(f"리뷰: '{리뷰}'")
    print(f"  나이브 베이즈:  {결과_nb}  (긍정 {확률_nb:.1%})")
    print(f"  로지스틱 회귀: {결과_lr}  (긍정 {확률_lr:.1%})")
    print()

새 리뷰 예측 결과:
리뷰: '치킨이 바삭하고 소스가 완벽해요 최고예요'
  나이브 베이즈:  긍정 😊  (긍정 75.2%)
  로지스틱 회귀: 긍정 😊  (긍정 63.7%)

리뷰: '배달이 세 시간이나 걸렸고 음식도 식어있었어요'
  나이브 베이즈:  부정 😞  (긍정 41.2%)
  로지스틱 회귀: 부정 😞  (긍정 47.5%)

리뷰: '맛은 보통이에요 특별하지는 않아요'
  나이브 베이즈:  부정 😞  (긍정 48.2%)
  로지스틱 회귀: 긍정 😊  (긍정 51.4%)

리뷰: '포장은 좋았는데 음식 맛이 별로였어요'
  나이브 베이즈:  부정 😞  (긍정 46.6%)
  로지스틱 회귀: 부정 😞  (긍정 49.8%)

리뷰: '가격 대비 양도 많고 배달도 빠르고 만족해요'
  나이브 베이즈:  긍정 😊  (긍정 67.4%)
  로지스틱 회귀: 긍정 😊  (긍정 60.1%)



In [27]:
# ✏️ 직접 입력해서 예측해보세요!
내_리뷰 = "여기에 원하는 리뷰를 입력하세요"

결과, 확률 = 예측하기(내_리뷰, 모델_LR)
print(f"리뷰: '{내_리뷰}'")
print(f"예측: {결과}  (긍정 확률: {확률:.1%})")

리뷰: '여기에 원하는 리뷰를 입력하세요'
예측: 긍정 😊  (긍정 확률: 56.8%)


> 💡 **현실 팁**  
> 확률이 0.4~0.6 사이로 애매하게 나오는 리뷰는 모델이 확신하지 못하는 것입니다.  
> 실제 서비스에서는 이런 경우를 별도로 분류해서 사람이 직접 검토하게 합니다.  
> 이를 **Human-in-the-loop** 방식이라고 합니다.

---
## 🏆 STEP 9. 도전 과제 — 영화 리뷰 감성 분석

**목표:** STEP 0의 핵심 코드를 영화 리뷰 데이터에 그대로 적용합니다.  
코드 구조는 완전히 동일합니다. **데이터만 바꾸면 됩니다.**

> 💡 **배달 vs 영화 비교**  
> 같은 파이프라인이지만 다른 가중치를 학습합니다.  
> 배달: "바삭", "빠르다" → 긍정 / "식다", "늦다" → 부정  
> 영화: "감동", "연기", "훌륭" → 긍정 / "지루", "조잡", "허망" → 부정  
> 이것이 머신러닝의 핵심 — **데이터가 바뀌면 모델이 스스로 새 패턴을 학습합니다.**

In [28]:
# 영화 리뷰 데이터
긍정_영화 = [
    "연기가 정말 훌륭하고 스토리도 감동적이에요",
    "배우들의 연기력이 뛰어나고 영상미도 아름다워요",
    "스토리 전개가 탄탄하고 반전도 훌륭했어요",
    "음악이 영화 분위기와 완벽하게 어울려요",
    "감독의 연출이 세밀하고 캐릭터도 잘 살렸어요",
    "두 시간이 전혀 지루하지 않았어요 최고의 영화",
    "몰입감이 대단하고 결말이 인상적이었어요",
    "배우 케미가 좋고 대사도 자연스러워요",
    "영상미가 뛰어나고 OST도 감동적이에요",
    "현실적인 스토리와 훌륭한 연기가 인상적이에요",
]

부정_영화 = [
    "시나리오가 진부하고 연기도 어색했어요",
    "스토리가 너무 예측 가능하고 지루했어요",
    "특수효과만 화려하고 내용이 없어요",
    "결말이 너무 허망하고 개연성이 부족해요",
    "두 시간이 너무 길게 느껴지고 집중이 안 됐어요",
    "배우 캐스팅이 잘못됐고 대사도 어색했어요",
    "스토리가 산으로 가고 결말도 이해가 안 돼요",
    "기대했는데 실망이에요 돈이 아까워요",
    "연출이 조잡하고 CG도 조악해요",
    "주인공 캐릭터가 매력 없고 공감이 안 돼요",
]

X_영화 = 긍정_영화 + 부정_영화
y_영화 = [1]*10 + [0]*10

print(f"영화 리뷰: {len(X_영화)}개 (긍정 {sum(y_영화)}개, 부정 {len(y_영화)-sum(y_영화)}개)")

영화 리뷰: 20개 (긍정 10개, 부정 10개)


In [29]:
# STEP 1: 전처리
X_영화 = [전처리(r) for r in X_영화]
print("전처리 완료!")
print("예시:", X_영화[0])

전처리 완료!
예시: 연기가 정말 훌륭하고 스토리도 감동적이에요


In [30]:
# STEP 2: 훈련/테스트 분리
X_tr, X_te, y_tr, y_te = train_test_split(
    X_영화, y_영화, test_size=0.25, random_state=42)

print(f"훈련: {len(X_tr)}개  /  테스트: {len(X_te)}개")

훈련: 15개  /  테스트: 5개


In [31]:
# STEP 3: TF-IDF 변환
변환기2 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_tr_vec = 변환기2.fit_transform(X_tr)
X_te_vec = 변환기2.transform(X_te)

print(f"특성 수: {X_tr_vec.shape[1]}개")

특성 수: 580개


In [32]:
# STEP 4: 두 모델 학습 + 예측
영화_NB = MultinomialNB()
영화_NB.fit(X_tr_vec, y_tr)
예측_NB2 = 영화_NB.predict(X_te_vec)

영화_LR = LogisticRegression(max_iter=1000, random_state=42)
영화_LR.fit(X_tr_vec, y_tr)
예측_LR2 = 영화_LR.predict(X_te_vec)

print("학습 완료!")

학습 완료!


In [33]:
# STEP 5: 성능 비교
print(f"나이브 베이즈:  {accuracy_score(y_te, 예측_NB2):.2%}  {'█'*int(accuracy_score(y_te, 예측_NB2)*20)}")
print(f"로지스틱 회귀: {accuracy_score(y_te, 예측_LR2):.2%}  {'█'*int(accuracy_score(y_te, 예측_LR2)*20)}")

나이브 베이즈:  60.00%  ████████████
로지스틱 회귀: 80.00%  ████████████████


In [34]:
# STEP 6: 상세 성능
print("로지스틱 회귀 상세:")
print(classification_report(y_te, 예측_LR2,
                              target_names=["부정","긍정"], zero_division=0))

로지스틱 회귀 상세:
              precision    recall  f1-score   support

          부정       1.00      0.50      0.67         2
          긍정       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5



In [35]:
# STEP 7 (심화): 영화 리뷰 긍정/부정 핵심 패턴 (필터 적용)
feat2 = 변환기2.get_feature_names_out()
coef2 = 영화_LR.coef_[0]

의미있는_idx2 = [i for i, p in enumerate(feat2) if 의미있는_패턴(p)]
필터_feat2   = feat2[의미있는_idx2]
필터_coef2   = coef2[의미있는_idx2]

print("영화 리뷰 — 긍정에 강한 패턴 TOP 8:")
for i in 필터_coef2.argsort()[-8:][::-1]:
    print(f"  '{필터_feat2[i]}': {필터_coef2[i]:+.3f}")
print()
print("영화 리뷰 — 부정에 강한 패턴 TOP 8:")
for i in 필터_coef2.argsort()[:8]:
    print(f"  '{필터_feat2[i]}': {필터_coef2[i]:+.3f}")

영화 리뷰 — 긍정에 강한 패턴 TOP 8:
  ' 훌륭': +0.108
  '훌륭': +0.108
  ' 인상적': +0.108
  '인상적이': +0.108
  '인상': +0.108
  ' 인상': +0.108
  '적이': +0.108
  '인상적': +0.108

영화 리뷰 — 부정에 강한 패턴 TOP 8:
  ' 너무': -0.135
  ' 너무 ': -0.135
  '너무': -0.135
  '너무 ': -0.135
  '리가 ': -0.102
  '리가': -0.102
  '스토리가': -0.102
  '토리가 ': -0.102


---
## 📌 전체 요약

### 오늘 배운 파이프라인

```
리뷰 텍스트
  ↓ ① 전처리()              특수문자 제거
  ↓ ② train_test_split()    훈련 75% / 테스트 25%
  ↓ ③ TfidfVectorizer       char_wb + ngram(2,4)
     .fit_transform(훈련)    훈련만 fit!
     .transform(테스트)      테스트는 transform만!
  ↓ ④ 모델.fit()            나이브 베이즈 or 로지스틱 회귀
     모델.predict()
  ↓ ⑤ classification_report()   정확도 + Precision + Recall + F1
```

### 핵심 개념 정리

| 개념 | 한 줄 설명 | 현실 팁 |
|------|-----------|--------|
| 레이블 | 모델에게 알려주는 정답 (1/0) | 레이블 품질이 성능의 절반 |
| 훈련/테스트 분리 | 보지 않은 데이터로 진짜 성능 측정 | 75:25 또는 80:20 |
| 데이터 누수 | 테스트 정보가 학습에 섞이는 반칙 | 성능이 너무 좋으면 의심 |
| char n-gram | 2~4글자 조합으로 특성 추출 | KoNLPy 없이도 한국어 잘 됨 |
| fit_transform | 훈련 데이터: 학습+변환 동시에 | 테스트엔 절대 fit 금지 |
| 나이브 베이즈 | 단어 확률로 분류 | 데이터 적을 때 먼저 시도 |
| 로지스틱 회귀 | 단어 가중치로 분류 | 가중치로 모델 디버깅 가능 |
| Precision | 예측의 정확성 | 오탐이 치명적일 때 중요 |
| Recall | 실제를 빠짐없이 찾기 | 미탐이 치명적일 때 중요 |
| F1 | 두 지표의 균형 | 일반적 상황에서 기본 지표 |

---

### 🗺️ 다음 주 예고 — 5주차: Word2Vec

```
오늘의 한계:
  "치킨"과 "닭고기" → 완전히 다른 표현으로 처리 (의미 유사성 모름)

다음 주:
  "치킨" ≈ "닭고기"  (벡터 공간에서 가까운 위치)
  "왕 - 남자 + 여자 = 여왕"  (단어 벡터 연산)
  단어의 '의미'를 숫자로 표현!
```